In [1]:
# rag_local.py
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader # Or UnstructuredPDFLoader

load_dotenv() # Optional: Loads environment variables from.env file

DATA_PATH = "PDFFiles/"
#PDF_FILENAME = "opsindoor.pdf" 

def load_documents():
    """Loads documents from the specified data path."""
    #pdf_path = os.path.join(DATA_PATH, PDF_FILENAME)
    #loader = PyPDFLoader(pdf_path)
    # loader = UnstructuredPDFLoader(pdf_path) # Alternative
    loader = DirectoryLoader(
    DATA_PATH,
    glob="*.pdf",
    loader_cls=PyPDFLoader,
    show_progress=True # Optional: shows a progress bar in the console
)

    documents = loader.load()
    print(f"Loaded {len(documents)} page(s) from {DATA_PATH}")
    return documents

# documents = load_documents() # Call this later

/Users/saumitrajoshi/pythonvenv/pythonvenv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents):
    """Splits documents into smaller chunks."""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
        length_function=len,
        is_separator_regex=False,
    )
    all_splits = text_splitter.split_documents(documents)
    print(f"Split into {len(all_splits)} chunks")
    return all_splits

# loaded_docs = load_documents()
# chunks = split_documents(loaded_docs) # Call this later

In [3]:
from langchain_ollama import OllamaEmbeddings

def get_embedding_function(model_name="nomic-embed-text"):
    """Initializes the Ollama embedding function."""
    # Ensure Ollama server is running (ollama serve)
    embeddings = OllamaEmbeddings(model=model_name)
    print(f"Initialized Ollama embeddings with model: {model_name}")
    return embeddings


In [4]:
from langchain_chroma import Chroma

CHROMA_PATH = "chroma_db" # Directory to store ChromaDB data

def get_vector_store(embedding_function, persist_directory=CHROMA_PATH):
    """Initializes or loads the Chroma vector store."""
    vectorstore = Chroma(
        persist_directory=persist_directory,
        embedding_function=embedding_function
    )
    print(f"Vector store initialized/loaded from: {persist_directory}")
    return vectorstore

embedding_function = get_embedding_function()
vector_store = get_vector_store(embedding_function) # Call this later

Initialized Ollama embeddings with model: nomic-embed-text
Vector store initialized/loaded from: chroma_db


In [5]:
def index_documents(chunks, embedding_function, persist_directory=CHROMA_PATH):
    """Indexes document chunks into the Chroma vector store."""
    print(f"Indexing {len(chunks)} chunks...")
    # Use from_documents for initial creation.
    # This will overwrite existing data if the directory exists but isn't a valid Chroma DB.
    # For incremental updates, initialize Chroma first and use vectorstore.add_documents().
    #vectorstore = Chroma.from_documents(
    #    documents=chunks,
    #    embedding=embedding_function,
    #   persist_directory=persist_directory
    #)

    vectorstore = get_vector_store(embedding_function, persist_directory)
    vectorstore.add_documents(chunks)
    #vectorstore.persist() # Ensure data is saved
    print(f"Indexing complete. Data saved to: {persist_directory}")
    return vectorstore

#... (previous function calls)


#vector_store = index_documents(chunks, embedding_function) # Call this for initial indexing

In [6]:
mbedding_function = get_embedding_function()
vector_store = Chroma(persist_directory=CHROMA_PATH, embedding_function=embedding_function)

Initialized Ollama embeddings with model: nomic-embed-text


In [7]:
# rag_local.py (continued)
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def create_rag_chain(vector_store, llm_model_name="qwen3:8b", context_window=8192):
    """Creates the RAG chain."""
    # Initialize the LLM
    llm = ChatOllama(
        model=llm_model_name,
        temperature=0, # Lower temperature for more factual RAG answers
        num_ctx=context_window # IMPORTANT: Set context window size
    )
    print(f"Initialized ChatOllama with model: {llm_model_name}, context window: {context_window}")

    # Create the retriever
    retriever = vector_store.as_retriever(
        search_type="mmr", # Or "mmr"
        search_kwargs={'k': 10} # Retrieve top 10 relevant chunks
    )
    print("Retriever initialized.")

    # Define the prompt template
    template = """Answer the question based ONLY on the following context:
{context}

Question: {question}
"""
    prompt = ChatPromptTemplate.from_template(template)
    print("Prompt template created.")

    # Define the RAG chain using LCEL
    rag_chain = (
        {"context": retriever, "question": RunnablePassthrough()}
| prompt
| llm
| StrOutputParser()
    )
    print("RAG chain created.")
    return rag_chain

#... (previous function calls)
vector_store = get_vector_store(embedding_function) # Assuming DB is already indexed
rag_chain = create_rag_chain(vector_store) # Call this later

Vector store initialized/loaded from: chroma_db
Initialized ChatOllama with model: qwen3:8b, context window: 8192
Retriever initialized.
Prompt template created.
RAG chain created.


In [9]:
def query_rag(chain, question):
    """Queries the RAG chain and prints the response."""
    print("\nQuerying RAG chain...")
    print(f"Question: {question}")
    response = chain.invoke(question)
    print("\nResponse:")
    print(response)

# --- Main Execution ---
if __name__ == "__main__":
    # 1. Load Documents
    docs = load_documents()

    # 2. Split Documents
    chunks = split_documents(docs)

    # 3. Get Embedding Function
    embedding_function = get_embedding_function() # Using Ollama nomic-embed-text

    # 4. Index Documents (Only needs to be done once per document set)
    # Check if DB exists, if not, index. For simplicity, we might re-index here.
    # A more robust approach would check if indexing is needed.
    print("Attempting to index documents...")
    vector_store = index_documents(chunks, embedding_function)
    # To load existing DB instead:
    # vector_store = get_vector_store(embedding_function)

    # 5. Create RAG Chain
    rag_chain = create_rag_chain(vector_store, llm_model_name="qwen3:8b") # Use the chosen Qwen 3 model

    # 6. Query
    query_question = "Create a summary of all judgements" # Replace with a specific question
    query_rag(rag_chain, query_question)

    #query_question_2 = "Summarize the introduction section." # Another example
    #query_rag(rag_chain, query_question_2)

    #query_question_3 = "what does the document say about parking? Who owns it? Society or Individual" # Another example
    #query_rag(rag_chain, query_question_3)

    

100%|█████████████████████████████████████████████| 4/4 [00:00<00:00,  4.96it/s]


Loaded 66 page(s) from PDFFiles/
Split into 141 chunks
Initialized Ollama embeddings with model: nomic-embed-text
Attempting to index documents...
Indexing 141 chunks...
Vector store initialized/loaded from: chroma_db
Indexing complete. Data saved to: chroma_db
Initialized ChatOllama with model: qwen3:8b, context window: 8192
Retriever initialized.
Prompt template created.
RAG chain created.

Querying RAG chain...
Question: Create a summary of all judgements

Response:
**Summary of Judgements from the Provided Documents:**

1. **Civil Appeal Cases (Documents 3395ef77-4d26-43fb-88e7-49c5c7bb8d7a & 0ba71654-0274-4dff-9557-42ec062ec225):**  
   - The Supreme Court dismissed appeals related to a **Consumer Protection Act** case (Civil Appeal Nos. 10527-10528 of 2024).  
   - The **landowners and developer** dispute over joint and several liability was unresolved, with previous orders directing the developer to pay delay compensation. The court rejected appeals, citing no merit in the argum

In [10]:
query_question_2 = "what do the judgements say about Sistema Shyam Teleservices Limited" # Another example
query_rag(rag_chain, query_question_2)



Querying RAG chain...
Question: what do the judgements say about Sistema Shyam Teleservices Limited

Response:
The provided judgments do not mention "Sistema Shyam Teleservices Limited" or any related references to the company. The documents discuss legal cases involving landowners, developers, criminal appeals, and procedural issues in court proceedings, but there is no explicit connection to Sistema Shyam Teleservices Limited.


In [11]:
query_question_3 = "Summarize the judgement for  Nilesh Koshti." # Another example
query_rag(rag_chain, query_question_3)



Querying RAG chain...
Question: Summarize the judgement for  Nilesh Koshti.

Response:
The judgment in **Criminal Appeal No. 5357 of 2025** (Nilesh Koshti v. State of Madhya Pradesh) upholds the High Court's decision to dismiss the appellant's appeal. The appellant was convicted under **Sections 302** (murder) and **201** (causing death by negligence) of the Indian Penal Code (IPC), 1860. He was sentenced to **life imprisonment** and a fine of ₹1,000 for the murder charge, and **seven years' rigorous imprisonment** with a ₹1,000 fine for the negligence charge. The Supreme Court allowed the appeal, affirming the High Court's dismissal of the appellant's challenge to the conviction. The judgment, delivered on **February 20, 2026**, concludes that the trial and appellate courts correctly upheld the conviction based on the evidence. Parties were directed to bear their own costs.
